# Leer los archivos

In [1]:
import os
import numpy as np
root_dir = r"../Spectra/FinalSamples"
dir_wln = os.path.join(root_dir, f"Level2/LVL2_Wavelengths.txt")
dir_spec = os.path.join(root_dir, f"Level2/LVL2_All_Spectra.txt")
with open(dir_spec, "r") as f:
    first_line = f.readline().strip()
column_names = first_line.split("\t")
all_Spectra = np.genfromtxt(dir_spec, skip_header=1)   
# data is shape (N, Files)
# to recover column M1_P1:
#spec_m1_p1 = all_Spectra[:, j]
wavelengths = np.loadtxt(dir_wln, delimiter="\t")
dir_concentraciones = os.path.join(root_dir, f"concentraciones.txt")
concentraciones = np.genfromtxt(dir_concentraciones, skip_header=1) 

## Configurar y ejecutar el modelo

In [23]:
import numpy as np
import matplotlib.pyplot as plt
from pymcr.mcr import McrAR
#from pymcr.constraints import ConstraintNonnegativity
from scipy.optimize import nnls
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

7283

In [34]:
# ————————————— Datos de entrada —————————————
# wavelengths: array shape (N_wvls,)
# intensity_matrix: array shape (N_wvls, n_samples)
# conc: array shape (n_samples, 2)  columns = [Na, Mg]

D                = all_Spectra  # (N_wvls, n_samples)
conc             = concentraciones    # (n_samples,2)

In [35]:
# ————————————— Separar en train/test —————————————
indices = np.arange(D.shape[1])
train_idx, test_idx = train_test_split(indices, train_size=0.7, random_state=0)

D_train = D[:, train_idx]          # (N_wvls, n_train)
D_test  = D[:, test_idx]           # (N_wvls, n_test)
C_train = conc[train_idx, :]      # (n_train, 2)
C_test  = conc[test_idx, :]       # (n_test,  2)

In [37]:
# ————————————— Configurar MCR-ALS —————————————
mcrA = McrAR(
    c_regr='NNLS',
    st_regr='NNLS'
)

mcrB = McrAR(
    c_regr='NNLS',
    st_regr='NNLS'
)

In [38]:

# ————————————— Modelo A: inicializar con C_train —————————————
mcrA.fit(D_train, C=C_train, c_first=True)

# Extraer ST_estimada
ST_A = mcrA.ST_         # shape (2, N_wvls)
C_train_A = mcrA.C_     # should match (n_train, 2)

# Para test: resolver C_test_pred de ST_A · C^T ≈ D_test
# D ≈ ST^T · C^T  ⇒  D_test^T (n_test, N) = C_test_pred (n_test,2) · ST_A  (2,N)
# We solve each sample i: ST_A^T x = D_test[:,i] via NNLS
C_test_A = np.zeros((D_test.shape[1], 2))
for i in range(D_test.shape[1]):
    # nnls solves A x = b with x>=0; we need ST_A.T shape (N,2), b = D_test[:,i]
    C_test_A[i], _ = nnls(ST_A.T, D_test[:, i])

ValueError: Incompatible dimensions. The first dimension of A is 52, while the shape of b is (7283,)